# AF2FS + IGEM Frozen-Parent — CPU Preflight

No GPU and no training. This notebook validates the current branch, contract tests, canonical AF2FS parent checkpoints, and all three dedicated IGEM static audits before any Kaggle GPU quota is spent.

Required Kaggle input: `af2-ffab2-all-seeds-all-stages-state.zip`.
Internet ON. Accelerator: None/CPU.


In [ ]:
from pathlib import Path
import importlib,json,os,shutil,subprocess,sys,time,zipfile

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')
os.chdir(WORK)

INDEX={}
for root,dirs,files in os.walk(INPUT):
    for filename in files:
        INDEX.setdefault(filename,[]).append(Path(root)/filename)

def one(name):
    matches=sorted(INDEX.get(name,[]))
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus tepat satu {name}; ditemukan {matches}')
    return matches[0]

state=one('af2-ffab2-all-seeds-all-stages-state.zip')
PRIOR=WORK/'af2fs-igem-preflight-source'
if PRIOR.exists():
    shutil.rmtree(PRIOR)
PRIOR.mkdir(parents=True)

wanted={f'AF2FS_seed{s}_result.json' for s in (42,123,2026)}
with zipfile.ZipFile(state,'r') as z:
    members=[]
    for info in z.infolist():
        base=Path(info.filename).name
        if base in wanted or base=='best.pt':
            members.append(info)
    found={Path(x.filename).name for x in members if Path(x.filename).name in wanted}
    if found!=wanted:
        raise RuntimeError(f'AF2FS result tidak lengkap: {found}')
    for info in members:
        z.extract(info,PRIOR)
print('SOURCE READY:',len(members),'members')


In [ ]:
BRANCH='codex/af2-igem-parent-confirmation'
REPO=WORK/'coffee-bean-detection'
os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git','clone','--depth','1','--branch',BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)
],cwd=WORK,check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],cwd=WORK,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],cwd=WORK,check=True)

for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()

COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH)
print('COMMIT:',COMMIT)

tests=[
    'tests/test_af2_parent_residual_audit_tolerance.py',
    'tests/test_af2_igem_parent_confirmation.py',
]
p=subprocess.run([sys.executable,'-m','pytest','-q',*tests],cwd=REPO,text=True,capture_output=True)
print(p.stdout)
if p.stderr:
    print(p.stderr)
if p.returncode:
    raise RuntimeError(f'CPU PREFLIGHT TESTS FAILED rc={p.returncode}')
print('CPU CONTRACT TESTS PASS')


In [ ]:
from coffee_detector.af2_spectral.audit import sha256
from coffee_detector.af2_parent_residual.igem_confirmation import (
    AUDIT_REVISION,
    run_af2_igem_parent_static_audit,
)

SEEDS=(42,123,2026)

def find_one(root,name):
    matches=sorted(root.rglob(name))
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus tepat satu {name}; ditemukan {matches}')
    return matches[0].resolve()

results={s:find_one(PRIOR,f'AF2FS_seed{s}_result.json') for s in SEEDS}
best_files=sorted(PRIOR.rglob('best.pt'))

parents={}
for s in SEEDS:
    payload=json.loads(results[s].read_text(encoding='utf-8'))
    if payload.get('arm')!='AF2FS' or int(payload.get('seed',-1))!=s:
        raise RuntimeError(f'Parent result seed {s} invalid')
    if payload.get('test_images_accessed') is not False:
        raise RuntimeError(f'Parent result seed {s} membuka test')
    expected=payload['checkpoint_sha256']
    matches=[p.resolve() for p in best_files if sha256(p)==expected]
    if not matches:
        raise FileNotFoundError(f'Parent seed {s} SHA {expected} tidak ditemukan')
    parents[s]=matches[0]

OUT=WORK/'af2-igem-cpu-preflight'
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

audits={}
for s in SEEDS:
    path=OUT/f'AF2FS_IGEM_static_seed{s}.json'
    audit=run_af2_igem_parent_static_audit(parents[s],path,device='cpu',image_size=64)
    failed={k:v for k,v in audit['gates'].items() if v is False and k!='test_accessed'}
    print(
        'SEED',s,
        'decision=',audit['decision'],
        'revision=',audit.get('audit_revision'),
        'failed=',failed,
        'control_box_diff=',audit['records']['AF2IGEM0']['active_box_max_abs_diff'],
        'control_score_diff=',audit['records']['AF2IGEM0']['active_score_max_abs_diff'],
        'candidate_box_diff=',audit['records']['AF2IGEM1']['active_box_max_abs_diff'],
        'candidate_score_diff=',audit['records']['AF2IGEM1']['active_score_max_abs_diff'],
    )
    if (
        audit.get('decision')!='PASS'
        or audit.get('audit_revision')!=AUDIT_REVISION
        or audit.get('training_authorized') is not True
        or audit.get('test_access_authorized') is not False
        or failed
    ):
        raise RuntimeError(f'CPU STATIC AUDIT seed {s} FAIL')
    audits[s]=audit

print('ALL THREE CPU STATIC AUDITS PASS')


In [ ]:
report={
    'format':'coffee_detector.af2_parent_residual.igem_cpu_preflight.v1',
    'branch':BRANCH,
    'commit':COMMIT,
    'audit_revision':AUDIT_REVISION,
    'seeds':list(SEEDS),
    'parents':{
        str(s):{
            'checkpoint_sha256':sha256(parents[s]),
            'parent_result_sha256':sha256(results[s]),
            'audit_decision':audits[s]['decision'],
        } for s in SEEDS
    },
    'contract_tests_passed':True,
    'all_static_audits_passed':True,
    'training_executed':False,
    'test_opened':False,
}
report_path=OUT/'af2fs_igem_cpu_preflight_report.json'
report_path.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
archive=Path(shutil.make_archive(
    str(WORK/'af2-igem-cpu-preflight-output'),'zip',
    root_dir=WORK,base_dir=OUT.name
))
print(json.dumps(report,indent=2))
print('PREFLIGHT ZIP:',archive,archive.stat().st_size,'bytes')
print('SAFE NEXT STEP: GPU training notebook only after this preflight PASSes.')
